# Feedforward classification of the NMIST data
### Advanced Deep Learning 2024
This notebook was written originally Jon Sporring (mailto:sporring@di.ku.dk) and heavily inspired by https://clay-atlas.com/us/blog/2021/04/22/pytorch-en-tutorial-4-train-a-model-to-classify-mnist.

We consider the Modified National Institute of Standards and Technology database of handwritten digits (MNIST): http://yann.lecun.com/exdb/mnist/

## Installs

On non-colab system, is usually good to make an environment and install necessary tools there. E.g., anaconda->jupyter->terminal create an environment, if you have not already, and activate it:
```
conda create -n adl python=3.9
conda activate adl
```
then install missing packages such as:
```
conda install ipykernel torch matplotlib torchmetrics scikit-image jpeg
conda install -c conda-forge segmentation-models-pytorch ipywidgets
```
and if you want to add it to jupyter's drop-down menu
```
ipython kernel install --user --name=adl
```
Now reload the jupyter-notebook's homepage and make a new or load an existing file. On colab, the tools have to be installed everytime

In [41]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False
if IN_COLAB:
    !pip3 install torch matplotlib torchmetrics scikit-image segmentation-models-pytorch

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as dset
from torchvision import datasets, transforms
from torchsummary import summary
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

## Set global device

In [43]:
# GPU
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('GPU State:', device)

GPU State: cpu


## Functions

In [44]:
def training_loop(model, loss, optimizer, loader, epochs, verbose=True, device=device):
    """
    Run training of a model given a loss function, optimizer and a set of training and validation data.
    """

    # Train
    for epoch in range(epochs):
        running_loss = 0.0

        for times, data in enumerate(loader):
            inputs, labels = data[0].to(device), data[1].to(device)
            #inputs = inputs.view(inputs.shape[0], -1)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Foward + backward + optimize
            outputs = model(inputs)
            loss_tensor = loss(outputs, labels)
            loss_tensor.backward()
            optimizer.step()

            # Print statistics
            running_loss += loss_tensor.item()
            if verbose:
                if times % 100 == 99 or times+1 == len(loader):
                    print('[%d/%d, %d/%d] loss: %.3f' % (epoch+1, epochs, times+1, len(loader), running_loss/2000))

In [45]:
def evaluate_model(model, loader, device=device):
    """
    Evaluate a model 'model' on all batches of a torch DataLoader 'data_loader'.

    Returns: the total number of correct classifications,
             the total number of images
             the list of the per class correct classification,
             the list of the per class total number of images.
    """

    # Test
    correct = 0
    total = 0
#------------------------------------------------------------------------------------       
    y_true = []
    y_pred = []
#------------------------------------------------------------------------------------   
    with torch.no_grad():
        for data in loader:
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            #inputs = inputs.view(inputs.shape[0], -1)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
#------------------------------------------------------------------------------------               
            y_true += labels
            y_pred += predicted
#------------------------------------------------------------------------------------   
    class_correct = [0 for i in range(10)]
    class_total = [0 for i in range(10)]

    with torch.no_grad():
        for data in loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            #inputs = inputs.view(inputs.shape[0], -1)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            c = (predicted == labels).squeeze()
            for i in range(labels.shape[0]):
                label = labels[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1

    return (correct, total, class_correct, class_total, y_true, y_pred)


## Main program

In [46]:
# Transform
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5,), (0.5,)),]
)

In [47]:
# Data
trainSet = datasets.MNIST(root='MNIST', download=True, train=True, transform=transform)
testSet = datasets.MNIST(root='MNIST', download=True, train=False, transform=transform)
trainLoader = dset.DataLoader(trainSet, batch_size=64, shuffle=True)
testLoader = dset.DataLoader(testSet, batch_size=64, shuffle=False)

In [48]:
# Model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size =(3, 3), stride =(1, 1)),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size =2, stride=2, padding=0, dilation=1, ceil_mode=False),
            nn.Conv2d (16, 32, kernel_size =(3, 3), stride =(1, 1)),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size =2, stride=2, padding=0, dilation=1, ceil_mode=False),
            nn.Flatten(start_dim=1, end_dim =-1),
            nn.Linear(in_features =800, out_features =10, bias=True),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, input):
        return self.main(input)


net = Net().to(device)
print(net)
summary(net, input_size=(1, 28, 28))

Net(
  (main): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=800, out_features=10, bias=True)
    (8): LogSoftmax(dim=1)
  )
)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 26, 26]             160
              ReLU-2           [-1, 16, 26, 26]               0
         MaxPool2d-3           [-1, 16, 13, 13]               0
            Conv2d-4           [-1, 32, 11, 11]           4,640
              ReLU-5           [-1, 32, 11, 11]               0
         MaxPool2d-6             [-1, 32, 5, 5]               0
       

In [49]:
# Parameters
epochs = 4
lr = 0.002
loss = nn.NLLLoss()
optimizer = optim.SGD(net.parameters(), lr=0.002, momentum=0.9)

# Train
print('Training on %d images' % trainSet.data.shape[0])
training_loop(net, loss, optimizer, trainLoader, epochs)
print('Training Finished.\n')

Training on 60000 images
[1/4, 100/938] loss: 0.091
[1/4, 200/938] loss: 0.120
[1/4, 300/938] loss: 0.138
[1/4, 400/938] loss: 0.154
[1/4, 500/938] loss: 0.167
[1/4, 600/938] loss: 0.178
[1/4, 700/938] loss: 0.189
[1/4, 800/938] loss: 0.199
[1/4, 900/938] loss: 0.207
[1/4, 938/938] loss: 0.210
[2/4, 100/938] loss: 0.007
[2/4, 200/938] loss: 0.014
[2/4, 300/938] loss: 0.021
[2/4, 400/938] loss: 0.028
[2/4, 500/938] loss: 0.034
[2/4, 600/938] loss: 0.041
[2/4, 700/938] loss: 0.046
[2/4, 800/938] loss: 0.053
[2/4, 900/938] loss: 0.058
[2/4, 938/938] loss: 0.061
[3/4, 100/938] loss: 0.006
[3/4, 200/938] loss: 0.011
[3/4, 300/938] loss: 0.016
[3/4, 400/938] loss: 0.020
[3/4, 500/938] loss: 0.025
[3/4, 600/938] loss: 0.029
[3/4, 700/938] loss: 0.034
[3/4, 800/938] loss: 0.038
[3/4, 900/938] loss: 0.043
[3/4, 938/938] loss: 0.044
[4/4, 100/938] loss: 0.004
[4/4, 200/938] loss: 0.009
[4/4, 300/938] loss: 0.013
[4/4, 400/938] loss: 0.017
[4/4, 500/938] loss: 0.020
[4/4, 600/938] loss: 0.025
[4/

In [ ]:
# Test
correct, total, class_correct, class_total, y_true, y_pred = evaluate_model(net, testLoader)

precision, recall, f1_score, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
matrix = confusion_matrix(y_true, y_pred, labels=[0,1,2,3,4,5,6,7,8,9])
print("confusion_matrix: (row = data, col = prediction)\n", matrix, "\n")

print('Accuracy of the network on the %d test images: %d %%' % (testSet.data.shape[0], (100*correct / total)))

def Score(TP, TN, FP, FN):
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * ((precision*recall)/(precision+recall))
    print(f'TP: {TP}')
    print(f'TN: {TN}')
    print(f'FP: {FP}')
    print(f'FN: {FN}')
    print(f'Accuracy: {accuracy}')
    print(f'precision: {precision}')
    print(f'recall: {recall}')
    print(f'F1: {f1}')
 
TP = [0] * 10
TN = [0] * 10
FP = [0] * 10
FN = [0] * 10    
for i in range(10):
    for j in range(10):
        if i == j:
            TP[i] = matrix[i][j]
            TN[i] = total - TP[i]
        else:
            FN[i] += matrix[i][j]
            FP[i] += matrix[j][i]          
            
for i in range(10):      
    print(f"\n{i}:") 
    Score(TP[i], TN[i], FP[i], FN[i])        
    # print('Accuracy of %d: %3f' % (i, (class_correct[i]/class_total[i])))

confusion_matrix: (row = data, col = prediction)
 [[ 972    0    1    0    0    0    2    3    2    0]
 [   0 1131    2    1    0    0    0    0    1    0]
 [   4    5 1011    3    2    0    0    5    2    0]
 [   1    0    0 1001    0    3    0    3    2    0]
 [   1    0    0    1  972    0    2    1    2    3]
 [   3    0    1   10    0  873    1    2    2    0]
 [   9    3    0    1    2    4  936    0    3    0]
 [   1    5   12   10    1    1    0  993    0    5]
 [  10    1    6   12    3    3    3    3  928    5]
 [   5    7    0    7    9    8    1    8    3  961]] 

Accuracy of the network on the 10000 test images: 97 %

0:
TP: 972
TN: 9028
FP: 34
FN: 8
Accuracy: 0.9958175662218681
precision: 0.9662027833001988
recall: 0.9918367346938776
F1: 0.9788519637462235

1:
TP: 1131
TN: 8869
FP: 21
FN: 4
Accuracy: 0.9975062344139651
precision: 0.9817708333333334
recall: 0.9964757709251101
F1: 0.9890686488850022

2:
TP: 1011
TN: 8989
FP: 22
FN: 21
Accuracy: 0.9957184108334163
precision: